# Demo 13 — Implementare $d(n)$ per Flanger e Chorus su chitarra acustica

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 44100
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
chitarra_ac, sr = al.load_audio(AUDIO / "acoustic.wav", sr=SR, seconds=5.0)
n_idx = np.arange(len(chitarra_ac))
t_ac = n_idx / sr

# 1. Flanger: d(n) = (D/2) * (1 - cos(2*pi*fd*n/Fs)), con D = 5 ms, fd = 1 Hz, a = 0.8
D_fl_max = 0.005 * sr
fd_fl = 1.0
wet_flanger, d_flanger = al.modulated_delay(chitarra_ac, sr, delay_ms=2.5, depth_ms=2.5, rate_hz=fd_fl, phase=-np.pi/2)
y_flanger = (chitarra_ac + 0.8 * wet_flanger) / 1.8

# 2. Chorus: d_k(n) = D1 + (D2 - D1) * (0.5 + r_k(n)), con r_k(n) casuale lento in [-0.5, 0.5)
rng_ch = np.random.default_rng(42)
b_lp_lfo, a_lp_lfo = signal.butter(2, 2.5 / (sr / 2.0), btype="low")
D1_ch, D2_ch = 0.010 * sr, 0.025 * sr
d_chorus_voci = []
y_chorus = chitarra_ac.copy()
for voce_k in range(2):
    rumore_lento = signal.filtfilt(b_lp_lfo, a_lp_lfo, rng_ch.uniform(-1.0, 1.0, len(chitarra_ac)))
    r_k = 0.48 * rumore_lento / max(np.max(np.abs(rumore_lento)), 1e-12)
    d_k = D1_ch + (D2_ch - D1_ch) * (0.5 + r_k)
    d_chorus_voci.append(d_k)
    wet_voce, _ = al.modulated_delay(chitarra_ac, sr, delay_curve=d_k)
    y_chorus = y_chorus + 0.6 * wet_voce
y_chorus = y_chorus / 2.2

al.compare_audio([("Originale (acoustic.wav)", chitarra_ac),
                  ("Flanger (sinusoidale 0–5 ms, 1 Hz)", y_flanger),
                  ("Chorus (2 voci casuali 10–25 ms)", y_chorus)], sr)

fig = go.Figure()
passo_plot = 100
fig.add_trace(go.Scatter(x=t_ac[::passo_plot], y=1000 * d_flanger[::passo_plot] / sr,
                         name="Flanger: d(n) sinusoidale"))
for idx_v, d_k in enumerate(d_chorus_voci, 1):
    fig.add_trace(go.Scatter(x=t_ac[::passo_plot], y=1000 * d_k[::passo_plot] / sr,
                             name=f"Chorus: voce {idx_v} casuale lenta"))
fig.update_xaxes(title_text="Tempo [s]")
fig.update_yaxes(title_text="Ritardo d(n) [ms]")
fig.update_layout(height=360, title="Confronto tra le leggi di ritardo d(n) di Flanger e Chorus")
fig.show()
